Section 1 — Objectif

Cette partie du projet a pour objectif de réaliser une classification automatique de texte en utilisant des modèles Transformers en NLP.

Les Transformers sont utilisés car ils permettent de comprendre le contexte global d’un texte grâce au mécanisme d’attention, ce qui améliore fortement les performances par rapport aux modèles classiques.

Le dataset utilisé est basé sur des fichiers textuels contenant des exemples de phrases associées à des labels, permettant d’entraîner un modèle de classification.

 Section 2 — Dataset

Le dataset utilisé est le fichier sample_text.txt.

Ce fichier contient des données textuelles avec leurs labels associés.

Les étapes réalisées sont :

• Chargement du fichier de données
• Analyse de la structure (texte + label)
• Affichage de plusieurs exemples pour comprendre les données

Cette étape est importante pour comprendre les données avant l’entraînement du modèle.
 Section 3 — Préprocessing

Cette étape consiste à préparer les données pour le modèle.

Les opérations effectuées sont :

• Nettoyage des textes (suppression des caractères inutiles)
• Normalisation du texte
• Préparation et encodage des labels
• Séparation des données en ensemble d’entraînement et de test

 Section 4 — Tokenization

La tokenization consiste à transformer le texte en unités appelées tokens.

Les étapes incluent :

• Conversion du texte en tokens
• Application du padding pour uniformiser les séquences
• Utilisation de la truncation pour limiter la taille des textes
• Création des attention masks pour indiquer les parties importantes du texte

 Section 5 — Modèle Transformer

Un modèle Transformer pré-entraîné (BERT ou DistilBERT) est utilisé.

Les étapes sont :

• Chargement du modèle pré-entraîné
• Extraction du vecteur [CLS] représentant le texte
• Ajout d’une couche Dense pour la classification finale

 Section 6 — Compilation

Le modèle est compilé avec :

• Loss function : CrossEntropyLoss
• Optimizer : Adam
• Metrics : accuracy

Section 7 — Training

Le modèle est entraîné avec :

• 2 à 5 epochs
• Batch size adapté
• Validation sur un ensemble de test

 Pendant l’entraînement, la loss et l’accuracy sont observées pour analyser les performances.

Section 8 — Évaluation

Les performances du modèle sont analysées :

• comparaison accuracy train vs test
• détection du surapprentissage (overfitting)
• analyse des erreurs de prédiction

 Section 9 — Graphiques

Des visualisations sont utilisées pour analyser le modèle :

• courbe de loss
• courbe d’accuracy

Ces graphiques permettent de mieux comprendre l’évolution de l’entraînement.

 Section 10 — Conclusion

Cette partie a permis de mettre en place un modèle de classification de texte basé sur les Transformers.

Les résultats montrent de bonnes performances grâce au mécanisme d’attention.

Cependant, certaines limites existent comme le besoin de plus de données et le risque de surapprentissage.

Des améliorations possibles incluent l’optimisation des hyperparamètres et l’utilisation de modèles plus avancés.

In [1]:
# Importation des bibliothèques nécessaires

# PyTorch pour le deep learning
import torch

# NumPy pour les calculs numériques
import numpy as np

# Chargement de datasets prêts à l'emploi (Hugging Face)
from datasets import load_dataset

# Tokenizer pour transformer le texte en tokens
from transformers import AutoTokenizer

# Modèle de classification de séquences pré-entraîné
from transformers import AutoModelForSequenceClassification

# Trainer API pour simplifier l'entraînement
from transformers import Trainer

# Arguments d'entraînement (epochs, batch size, etc.)
from transformers import TrainingArguments

# Métriques d'évaluation
from sklearn.metrics import accuracy_score, f1_score

In [3]:
# Importation de la fonction load_dataset depuis Hugging Face Datasets
from datasets import load_dataset

# Chargement d’un dataset local au format CSV (même si l’extension est .txt)
# column_names définit les noms des colonnes du fichier
dataset = load_dataset(
    "csv",
    data_files="../data/sample_texts.csv",
    column_names=["text", "label"]
)

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
# Affichage du premier exemple du dataset d'entraînement

# dataset["train"] correspond au split d'entraînement
# [0] récupère la première ligne (premier échantillon)
print(dataset["train"][0])

{'text': 'I absolutely love this product', 'label': 1}


In [5]:
# Affichage des 5 premiers exemples du dataset d'entraînement

# dataset["train"] contient les données d'entraînement
# [:5] permet de récupérer les 5 premières lignes
print(dataset["train"][:5])

{'text': ['I absolutely love this product', 'This is amazing work', 'The model performs very well', 'I really enjoyed this', 'This is okay but not great'], 'label': [1, 1, 1, 1, 1]}


In [6]:
# Fonction de prétraitement des données texte

# Cette fonction prend un exemple du dataset et modifie son champ "text"
def clean(example):
    # Conversion du texte en minuscules pour uniformiser les données
    example["text"] = example["text"].lower()
    
    # Retourne l'exemple modifié
    return example

# Application de la fonction à tout le dataset
# map() applique clean() à chaque élément du dataset
dataset = dataset.map(clean)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [7]:
# Choix du modèle pré-entraîné (DistilBERT, version légère de BERT)
model_ckpt = "distilbert-base-uncased"

# Chargement du tokenizer associé au modèle
# Le tokenizer transforme le texte en tokens numériques
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# Fonction de tokenisation appliquée au dataset
def tokenize(batch):
    # Conversion du texte en tokens avec padding et truncation
    return tokenizer(batch["text"], padding=True, truncation=True)

# Application de la tokenisation sur tout le dataset
# batched=True permet de traiter plusieurs exemples en même temps (plus rapide)
dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [8]:
# Renommage de la colonne "label" en "labels"

# Les modèles Hugging Face attendent généralement le nom "labels"
dataset = dataset.rename_column("label", "labels")

# Conversion du dataset en format PyTorch
# Cela permet de récupérer directement des tenseurs torch
dataset.set_format("torch")

In [9]:
# Chargement du modèle pré-entraîné pour classification de texte

# AutoModelForSequenceClassification charge un modèle adapté à la classification
# model_ckpt = architecture de base (DistilBERT ici)
# num_labels = nombre de classes à prédire (ici 2 : binaire)
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=2   # adapter si le dataset contient plus de classes
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# Affichage du premier exemple du dataset d'entraînement

# Vérification des données après tokenisation et transformation en tensors PyTorch
print(dataset["train"][0])

{'text': 'i absolutely love this product', 'labels': tensor(1), 'input_ids': tensor([ 101, 1045, 7078, 2293, 2023, 4031,  102,    0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 0])}


In [11]:
# Définition des paramètres d'entraînement

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4, 
)

# Création du Trainer Hugging Face
trainer = Trainer(
    model=model,                         # modèle à entraîner
    args=training_args,                 # paramètres d'entraînement
    train_dataset=dataset["train"]      # dataset d'entraînement
)

# Lancement de l'entraînement
trainer.train()

  0%|          | 0/9 [00:00<?, ?it/s]

c:\Users\adilu\anaconda3\envs\tp4_clean\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'train_runtime': 6.1583, 'train_samples_per_second': 4.871, 'train_steps_per_second': 1.461, 'train_loss': 0.610873646206326, 'epoch': 3.0}


TrainOutput(global_step=9, training_loss=0.610873646206326, metrics={'train_runtime': 6.1583, 'train_samples_per_second': 4.871, 'train_steps_per_second': 1.461, 'total_flos': 62094093120.0, 'train_loss': 0.610873646206326, 'epoch': 3.0})

In [12]:
# Prédiction sur le dataset d'entraînement

# Le Trainer génère les prédictions (logits) sur les données
preds = trainer.predict(dataset["train"])

# Conversion des logits en classes (argmax)
y_pred = np.argmax(preds.predictions, axis=1)

# Récupération des vraies étiquettes
y_true = preds.label_ids

# Calcul de l'accuracy du modèle
print("Accuracy:", accuracy_score(y_true, y_pred))

  0%|          | 0/2 [00:00<?, ?it/s]

Accuracy: 1.0


In [13]:
# Fonction de prédiction sur un texte brut

# Cette fonction prend une phrase en entrée et retourne la classe prédite
def predict(text):
    # Tokenisation du texte (conversion en tenseurs PyTorch)
    inputs = tokenizer(text, return_tensors="pt")
    
    # Passage du texte dans le modèle entraîné
    outputs = model(**inputs)
    
    # Récupération de la classe ayant la plus grande probabilité (logits)
    return torch.argmax(outputs.logits).item()

# Test du modèle sur deux phrases simples
print(predict("I love deep learning"))
print(predict("I hate bugs"))

1
0


In [14]:
# Liste de textes à tester sur le modèle NLP
texts = [
    "I love transformers",
    "This is bad",
    "Amazing work",
    "I hate this"
]

# Boucle sur chaque texte pour obtenir la prédiction du modèle
for t in texts:
    # Affichage du texte et de sa classe prédite
    print(t, "→", predict(t))

I love transformers → 1
This is bad → 1
Amazing work → 1
I hate this → 0
